# Experiment: Agent OS Prompt Pack Audit

Objective:
- Verify that the prompt surface now matches the deployed Agent OS foundation.
- Audit the six locked agents, their task domains, policy boundaries, memory partitions, eval metrics, and workflow order from exported repo truth.

Success criteria:
- Exported prompt pack contains exactly Brandyn, Jordyn, Kobe, Oracle, Titan, and Maestro.
- Workflow order is `Brandyn -> Jordyn -> Kobe -> Oracle -> Titan` with Maestro as orchestrator.
- Each agent shows distinct allowed and denied capabilities with no role ambiguity.


In [ ]:
from __future__ import annotations

import json
import subprocess
from pathlib import Path

REPO = Path.cwd()
for candidate in [REPO, *REPO.parents]:
    if (candidate / 'package.json').exists() and (candidate / 'packages' / 'agent-os').exists():
        REPO = candidate
        break
EXPORT_PATH = REPO / 'output' / 'agent-os' / 'prompt-pack.json'
cmd = ['pnpm', 'agent-os:prompt-pack:export', str(EXPORT_PATH)]
result = subprocess.run(cmd, cwd=REPO, capture_output=True, text=True, check=True)
print(result.stdout.strip())
EXPORT_PATH.exists()


In [ ]:
bundle = json.loads(EXPORT_PATH.read_text())
summary = {
    'generatedAt': bundle['generatedAt'],
    'agentCount': bundle['agentCount'],
    'workflowOrder': bundle['workflowOrder'],
    'orchestrator': bundle['orchestrator'],
}
summary


## Agent Inventory

This cell flattens the exported prompt pack into the contract that prompts and evals are expected to follow.


In [ ]:
rows = []
for record in bundle['prompts']:
    rows.append({
        'agentId': record['agentId'],
        'taskDomain': record['taskDomain'],
        'workflowRole': record['workflowRole'],
        'allowedCount': len(record['allowedCapabilities']),
        'deniedCount': len(record['deniedCapabilities']),
        'memoryCollections': len(record['ownedMemoryCollections']),
        'evalMetrics': len(record['evalMetrics']),
        'handoffTargets': ', '.join(record['handoffTargets']) or '-',
        'delegationTargets': ', '.join(record['delegationTargets']) or '-',
    })
rows


## Prompt Audit Questions

- Does each prompt stay inside one enforceable task domain?
- Does each prompt reflect the denied capabilities that now exist in code?
- Does Maestro route rather than perform specialist work?
- Does the handoff chain stop after Titan rather than looping into undefined roles?


In [ ]:
for record in bundle['prompts']:
    print(f"\n[{record['displayName']}] domain={record['taskDomain']} role={record['workflowRole']}")
    print('  allowed:', ', '.join(record['allowedCapabilities']))
    print('  denied:', ', '.join(record['deniedCapabilities']))
    print('  memory:', ', '.join(record['ownedMemoryCollections']))
    print('  evals:', ', '.join(record['evalMetrics']))
    print('  handoff:', ', '.join(record['handoffTargets']) or '-')
    print('  delegation:', ', '.join(record['delegationTargets']) or '-')


## Next steps

- Re-run this notebook whenever Agent OS policy, memory, routing, or eval contracts change.
- Treat the exported prompt pack as the source of truth for future prompt and notebook updates.
- If a new named agent is added, fail this audit until the prompt pack, manifest, and notebook are updated together.
